# Experiment 1

Idea: With a given probability randomly allocate price within the range otherwise give
best action as suggested by policy

In [1]:
import copy

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from stable_baselines3.ppo import PPO

from config import (
    CALENDAR_SEED,
    DEMAND_MODEL_PRICE_GRANULARITY,
    FIGURES_PATH,
    N_SNAP_DAYS,
    NOISE_RANGE,
    NUM_PRODUCTS,
    NUM_TIMESTEPS,
    P_DIFF,
    P_MAX,
    P_MIN,
    PPO_HYPERPARAMS,
    PPO_TRAINING_EPISODES,
    RANDOM_PROBS_LIST,
    SEED_LIST,
    K,
)
from fair_dynamic_pricing_with_rl.utils.demand_model import DemandModelEstimator
from fair_dynamic_pricing_with_rl.utils.env import FMCGEnv
from fair_dynamic_pricing_with_rl.utils.experiment_utils import argmedian
from fair_dynamic_pricing_with_rl.utils.fairness_utils import compute_jains_index


In [2]:
unit_sales = pd.read_csv("../data/sales_train_evaluation.csv")
prices = pd.read_csv("../data/sell_prices.csv")
calendar = pd.read_csv("../data/calendar.csv")

In [3]:
demand_model_estimator = DemandModelEstimator(
    unit_sales_df = unit_sales,
    prices_df = prices,
    calendar_df = calendar,
    price_granularity = DEMAND_MODEL_PRICE_GRANULARITY,
    verbose = True
)

demand_model_parameters = demand_model_estimator.run_pipeline()

/Users/kisslucasara/Documents/levi-research-project/newest_codes/fair-dynamic-pricing-with-rl/src/fair_dynamic_pricing_with_rl/utils/demand_model.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  q = foods_demand.groupby(by=["item_id"]).mean().reset_index()


--- Fitting demand model for FOODS_1_096 ---
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.625
Model:                            OLS   Adj. R-squared:                  0.623
Method:                 Least Squares   F-statistic:                     308.1
Date:                Sat, 15 Aug 2026   Prob (F-statistic):          3.47e-270
Time:                        10:19:18   Log-Likelihood:                -3388.3
No. Observations:                1300   AIC:                             6793.
Df Residuals:                    1292   BIC:                             6834.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------

/Users/kisslucasara/Documents/levi-research-project/newest_codes/fair-dynamic-pricing-with-rl/src/fair_dynamic_pricing_with_rl/utils/demand_model.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  q_long = q.reset_index().melt(


In [4]:
# Register the environment so we can create it with gym.make()
gym.register(
    id="gymnasium_env/FMCGEnv",
    entry_point=FMCGEnv,
    max_episode_steps=500,  # Prevent infinite episodes
)

In [5]:
def running_model(env1,
                  alg,
                  T,
                  K,
                  num_products,
                  random_action_prob=0.5,
                  seed=None
                  ):
    """ PPO algorithm with some probability of choosing an action randomly.

    Args:
        env1 (FMCGEnv): the defined gym environment
        alg (stable_baselines3.PPO): PPO policy
        T (int): number of timesteps per episode
        K (int): number of episodes
        num_products (int): number of products
        random_action_prob (float): probability of choosing random action (in [0.0-1.0])
        seed (int): seed for reproducibility

    Returns:
        rew (np.ndarray): rewards (T,K)
        action_hist (np.ndarray): action history (T,K,L)
        groups (np.ndarray): groups according to binary sensitive attribute (T,K)
    """

    env = copy.deepcopy(env1)

    # dedicated RNG for random action override plus seeding action sampler
    rng = np.random.default_rng(seed)
    env.action_space.seed(seed)

    rew = np.zeros((T,K))
    action_hist = np.zeros((T,K,num_products))
    groups = np.zeros((T,K))

    for k in range(K):
        # sample initial state
        # seed each reset deterministically but distinctly per episode
        observation, info = env.reset(seed=None if seed is None else seed + k)

        for t in range(T):
            # save the group wrt. sensitive attribute
            groups[t,k] = observation['sensitive_attr']
            
            # take action using trained model
            action, _states = alg.predict(observation, deterministic=False)
            a = np.asarray(action)
            
            # with random_action_prob override with a random action
            if rng.random() < random_action_prob:
                a = env.action_space.sample()
                a = np.asarray(a)
            
            action_hist[t,k] = a

            # record new state/reward from action played
            observation, reward, terminated, truncated, info = env.step(a)

            rew[t,k] = reward
        
    return rew, action_hist, groups

In [6]:
# storing expected mean episode rewards
episode_rewards_comparison = np.zeros((K*len(RANDOM_PROBS_LIST),3))
# storing Jain's indexes and mean episode revenues
results = pd.DataFrame(np.zeros((len(RANDOM_PROBS_LIST),5)),columns=["mean_episode_revenue","jains_index","random_action_prob", "nonsnap_price", "snap_price"])

action_histories = {}
all_rewards = {}

rewards_over_seeds = np.zeros((len(SEED_LIST),NUM_TIMESTEPS,K))
action_hist_over_seeds = np.zeros((len(SEED_LIST),NUM_TIMESTEPS,K,NUM_PRODUCTS))
groups_over_seeds = np.zeros((len(SEED_LIST),NUM_TIMESTEPS,K))

for idx in range(len(RANDOM_PROBS_LIST)):
    print(f"RUN STARTED FOR RANDOM PROB. = {RANDOM_PROBS_LIST[idx]}")
    for seed_idx in range(len(SEED_LIST)):
        print(f"RUN STARTED FOR SEED {SEED_LIST[seed_idx]}")
        env = gym.make("gymnasium_env/FMCGEnv",
               p_min=P_MIN,
                p_max=P_MAX,
                p_diff=P_DIFF,
                betas=[params.to_numpy() for params in demand_model_parameters[3]],
                max_demand=demand_model_parameters[4],
                n_snap_days=N_SNAP_DAYS,
                calendar_seed=CALENDAR_SEED,
                noise_range=NOISE_RANGE,
                T=NUM_TIMESTEPS
               )

        # define and warm-up algorithm
        # PPO hyperparameters from Liu et al. (2024)
        model = PPO(policy="MultiInputPolicy",
                    env=env,
                    learning_rate=PPO_HYPERPARAMS["learning_rate"],
                    batch_size=PPO_HYPERPARAMS["batch_size"], # default
                    clip_range=PPO_HYPERPARAMS["clip_range"], # default
                    seed=SEED_LIST[seed_idx],
                    ent_coef=PPO_HYPERPARAMS["ent_coef"])

        model.learn(total_timesteps=PPO_TRAINING_EPISODES*NUM_TIMESTEPS,
                    progress_bar=True)

        rewards, action_hist, groups = running_model(env1=env,
                                                alg=model,
                                                T=NUM_TIMESTEPS,
                                                K=K,
                                                num_products=NUM_PRODUCTS,
                                                random_action_prob=RANDOM_PROBS_LIST[idx],
                                                seed=SEED_LIST[seed_idx])
        rewards_over_seeds[seed_idx, ...] = rewards
        action_hist_over_seeds[seed_idx, ...] = action_hist
        groups_over_seeds[seed_idx, ...] = groups

    # save _over_seeds arrays for further analysis
    np.save(f"../output/experiment_1/rewards_over_seeds_{RANDOM_PROBS_LIST[idx]}.npy",
            rewards_over_seeds)
    np.save(f"../output/experiment_1/action_hist_over_seeds_{RANDOM_PROBS_LIST[idx]}.npy",
                action_hist_over_seeds)
    np.save(f"../output/experiment_1/groups_over_seeds_{RANDOM_PROBS_LIST[idx]}.npy",
                groups_over_seeds)

    # choose based on median of last episode reward
    last_episode_rewards_over_seeds = rewards_over_seeds.sum(axis=1)[:,-1]
    median_seed_idx = argmedian(last_episode_rewards_over_seeds)

    rewards_chosen = rewards_over_seeds[median_seed_idx,...]
    action_hist_chosen = action_hist_over_seeds[median_seed_idx,...]
    groups_chosen = groups_over_seeds[median_seed_idx,...]

    # save action histories & rewards for further analysis
    action_histories[RANDOM_PROBS_LIST[idx]] = action_hist_chosen
    all_rewards[RANDOM_PROBS_LIST[idx]] = rewards_chosen
 
    episode_rewards = rewards_chosen.sum(axis=0)
    e_mean_episode_reward = episode_rewards.cumsum() / (np.arange(K) + 1)

    # expected mean episode rewards
    episode_rewards_comparison[(idx*K):((idx+1)*K),0] = np.arange(K)
    episode_rewards_comparison[(idx*K):((idx+1)*K),1] = e_mean_episode_reward

    # random action probability
    episode_rewards_comparison[(idx*K):((idx+1)*K),2] = RANDOM_PROBS_LIST[idx]

    # save main performance metrics
    results.loc[idx,"random_action_prob"] = RANDOM_PROBS_LIST[idx]
    results.loc[idx,"mean_episode_revenue"] = episode_rewards.mean()
    results.loc[idx, "jains_index"], group_values = compute_jains_index(action_hist_chosen,
                                                                        groups_chosen,
                                                                        p_min=P_MIN,
                                                                        p_diff=P_DIFF)
    results.loc[idx, "nonsnap_price"] = group_values[False]
    results.loc[idx, "snap_price"] = group_values[True]
    print(f"Iteration done with mean episode revenue of {episode_rewards.mean()} and Jain's index of {results.loc[idx, "jains_index"]}.")
    print(f"Average price differences: non-SNAP: {group_values[False]} vs. SNAP: {group_values[True]}")


RUN STARTED FOR RANDOM PROB. = 0.0
RUN STARTED FOR SEED 1024


Output()

Output()

RUN STARTED FOR SEED 1224


Output()

...

Output()

RUN STARTED FOR SEED 1224


Output()

RUN STARTED FOR SEED 1424


Iteration done with mean episode revenue of 795.7151511699308 and Jain's index of 0.9999995161301084.
Average price differences: non-SNAP: 13.99689375 vs. SNAP: 14.016380000000002


In [ ]:
# save & print results
results.to_csv("../output/experiment_1/results.csv",
               index=False)
results 

,mean_episode_revenue,jains_index,random_action_prob,nonsnap_price,snap_price
0,9944.110615,0.999986,0.000000,16.586586,16.463847
1,7432.007857,0.999984,0.111111,16.159881,16.031368
2,6234.412375,0.999981,0.222222,15.906685,15.769335
3,5403.368747,0.999982,0.333333,15.657278,15.525910
4,4677.122458,0.999992,0.444444,15.465479,15.379423
5,3222.220481,0.999990,0.555556,15.068998,14.975790
6,2555.864505,0.999995,0.666667,14.815056,14.750685
7,2022.275501,0.999999,0.777778,14.551205,14.523243
8,1404.169498,0.999994,0.888889,14.303495,14.232620
9,795.715151,1.000000,1.000000,13.996894,14.016380
